In [33]:
pip install pandas openpyxl bertopic sentence-transformers umap-learn hdbscan scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [34]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer

In [17]:
# 1 读取新闻数据
# =========================
file_path = r"/Users/lixiangzhen/Desktop/li/cityu/semA /6601/6601 新闻网站数据.xlsx"

df = pd.read_excel(file_path)

print("原始新闻数量:", len(df))


原始新闻数量: 4760


In [35]:
# 2 文本清洗（含：去网址、日期归一化、去新闻模板、去英文数字）
# =========================

# 新闻页脚/模板短语（去掉后能减少“责任编辑”“纠错”等无关主题词）
BOILERPLATE_PATTERNS = [
    r"责任编辑[^\s]*",
    r"纠错[^\s]*",
    r"图片来源[^\s]*",
    r"阅读下一篇[^\s]*",
    r"澎湃新闻[^\s]*",
    r"点击进入[^\s]*",
    r"原文链接[^\s]*",
]

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)       # 删除网址
    # 先做日期归一：把“2024年3月7日”“3月7日”等统一换成占位符，避免删数字后变成“年月日”污染主题
    text = re.sub(r"\d{4}年\d{1,2}月\d{1,2}日?", "[日期]", text)
    text = re.sub(r"\d{1,2}月\d{1,2}日?", "[日期]", text)
    text = re.sub(r"今年\d{1,2}月", "[日期]", text)
    text = re.sub(r"去年\d{1,2}月", "[日期]", text)
    for p in BOILERPLATE_PATTERNS:
        text = re.sub(p, "", text)
    text = re.sub(r"[A-Za-z0-9]", "", text)  # 删除英文数字（在日期归一之后）
    text = re.sub(r"\[日期\]", "", text)     # 去掉占位符，不参与主题词
    text = re.sub(r"\s+", "", text)          # 删除空格
    return text

df["clean_text"] = df["Abstract"].apply(clean_text)

In [36]:
# 3 新闻筛选
# workplace + mental health
# =========================

work_terms = [
"工作","职场","公司","员工","上班","企业","单位",
"岗位","职业","劳动","白领","加班","同事"
]

mental_terms = [
"焦虑","抑郁","心理","压力","情绪","心理健康",
"精神","心理问题","心理疾病","倦怠","疲惫",
"心理咨询","心理治疗","崩溃","头晕","嗜睡",
]

work_pattern = "|".join(work_terms)
mental_pattern = "|".join(mental_terms)

df_filtered = df[
    df["clean_text"].str.contains(work_pattern) &
    df["clean_text"].str.contains(mental_pattern)
]

print("筛选后新闻数量:", len(df_filtered))

docs = df_filtered["clean_text"].tolist()


筛选后新闻数量: 2039


In [37]:
# 4 中文新闻 stopwords（扩充：模板词、日期残留、地点、政策、衔接词，减少无关主题词）
# =========================

stopwords = [
    "表示","指出","认为","介绍","记者","近日","建立",
    "同时","此外","其中","通过","根据","相关","进行","服务",
    "已经","正在","一个","一些","一种","这个","搭建","他认为",
    "我们","他们","她们","可以","可能","需要","践行","值得注意的是",
    "比如","例如","因此","如果","因为","构建","文化","老年人",
    "所以","如今","近年来","当前","此次","进一步","这些",
    "方面","情况","建设","然而","注意","打造","作家","编者",
    "了解","强调","对我来说","下一步","以下","随后","我想",
    # 新闻模板/页脚
    "责任编辑","纠错","图片来源","阅读下一篇","澎湃新闻","来源","以下简称","编辑","编者","化名",
    # 日期相关
    "年月日","月日","今年月","月日下午","月日晚","年月","去年月","月份","下午", "等等", "围绕","不久后", "况且","左右","分钟","今年",
    # 地点/媒体
    "北京","上海","街道","活动现场","为主题","活动中","全会","彼得","活动","政府",
    # 政策/经济
    "十四五","十五五","亿元","万元","同比增长","到年","年初","今年以来","三中全会","党","价值","开展","四中全会","评选",
    "党的二十届四中全会通过的",
    # 衔接/套话
    "据了解","另一方面","于是","他说","她认为","有时","其次","在这里","主题",
    "由此","展望未来","首先","相反","总之","并且","指出","但是","最后",
    "面对面","走进家庭","模式","尽管如此","就这样","为此","而是","她说",
    "未来","另外","与此同时","一书","不过","这种","当然","那么","实现",
    "内容","机构","作为","事实上","实际上","目前","注意","一方面","值得一提的是",
]

# =========================
# 5 Vectorizer
# =========================

vectorizer_model = CountVectorizer(
    stop_words=stopwords,
    ngram_range=(1, 2),
    min_df=3,
)


In [38]:
pip install bertopic pandas openpyxl sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [39]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

In [40]:
# 6 Embedding 模型
# =========================

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)


In [41]:
# 7 BERTopic 建模
# =========================

topic_model = BERTopic(

    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",

    calculate_probabilities=False,   # 避免 reduce bug

    min_topic_size=40,

    verbose=True
)

topics, _ = topic_model.fit_transform(docs)


2026-03-09 01:50:58,590 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

2026-03-09 01:51:08,979 - BERTopic - Embedding - Completed ✓
2026-03-09 01:51:08,980 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-09 01:51:11,980 - BERTopic - Dimensionality - Completed ✓
2026-03-09 01:51:11,981 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-09 01:51:12,031 - BERTopic - Cluster - Completed ✓
2026-03-09 01:51:12,033 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-09 01:51:12,144 - BERTopic - Representation - Completed ✓


In [42]:
# 8 查看初始 topics
# =========================

topic_info = topic_model.get_topic_info()


topic_info_initial = topic_info.copy()
topic_words_initial = {
    tid: topic_model.get_topic(tid)
    for tid in topic_info["Topic"] if tid >= 0 and topic_model.get_topic(tid)
}

print(topic_info)


   Topic  Count                   Name  \
0     -1    580     -1_生活_他指出_相比之下_她指出   
1      0    644     0_缓解工作压力_快递员_安全_线上   
2      1    459     1_职业倦怠_内卷_研究_工作压力大   
3      2    252  2_工作压力大_生活节奏快_焦虑_工作压力   
4      3     53          3_时间_身体_情感_可是   
5      4     51          4_抑郁_当时_此后_配合   

                                      Representation  \
0   [生活, 他指出, 相比之下, 她指出, 的状态, 截至目前, 科技, 精神, 现象, 的问题]   
1    [缓解工作压力, 快递员, 安全, 线上, 心理, 期间, 近期, 家庭矛盾, 行动, 品牌]   
2    [职业倦怠, 内卷, 研究, 工作压力大, 平台, 失眠, 但事实上, 下称, 项目, 焦虑]   
3  [工作压力大, 生活节奏快, 焦虑, 工作压力, 职业倦怠, 状态, 对此, 打工人, 抑郁...   
4    [时间, 身体, 情感, 可是, 成为, 编者按, 不仅如此, 你看, 在这个意义上, 海报]   
5   [抑郁, 当时, 此后, 配合, 记者采访发现, 很多时候, 聊天, 律师, 几天后, 慢下来]   

                                 Representative_Docs  
0  [，“学术原创力与构建中国自主知识体系的青年自觉”研讨会暨《探索与争鸣》年发展座谈会在中国浦...  
1  [近日，青海省海东市平安区纪委监委组织干部开展“廉韵传香·‘香’约纪委”香熏制作活动。此次活...  
2  [“预计年的股市场仍将温和上涨，而其中不乏结构性机会。”，博道基金董事长莫泰山在其撰写的新年...  
3  [这一年龄段的糖尿病患病率约为%-%，这一年龄段的人群通常处于代谢综合征的高发期，肥胖、高血...  
4  [北京｜不断不舍不离：个大叔和他们的身外物——欧阳应霁新书《不断不舍

In [26]:
# 9 Reduce topics
# =========================


topic_model = topic_model.reduce_topics(docs, nr_topics=5)

# 重新计算每篇文档所属主题
topics_reduced, _ = topic_model.transform(docs)

topic_info_reduced = topic_model.get_topic_info()

print("最终 topics:")
print(topic_info_reduced)


2026-03-08 23:11:40,807 - BERTopic - Topic reduction - Reducing number of topics
2026-03-08 23:11:40,807 - BERTopic - Topic reduction - Number of topics (5) is equal or higher than the clustered topics(4).
2026-03-08 23:11:40,808 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-08 23:11:41,045 - BERTopic - Representation - Completed ✓


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

2026-03-08 23:11:51,332 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-08 23:11:51,339 - BERTopic - Dimensionality - Completed ✓
2026-03-08 23:11:51,339 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-08 23:11:51,382 - BERTopic - Cluster - Completed ✓


最终 topics:
   Topic  Count                   Name  \
0     -1    411       -1_他指出_过去_期间_编者按   
1      0    690      0_缓解工作压力_高效_安全_问题   
2      1    682       1_职业倦怠_身体_的状态_此后   
3      2    256  2_工作压力大_生活节奏快_焦虑_工作压力   

                                      Representation  \
0  [他指出, 过去, 期间, 编者按, 她指出, 相比之下, 截至目前, 发布, 语言, 年轻...   
1    [缓解工作压力, 高效, 安全, 问题, 截至目前, 心理, 本次, 期间, 他指出, 第一]   
2     [职业倦怠, 身体, 的状态, 此后, 编者按, 家庭, 研究, 最终, 成为, 不仅如此]   
3  [工作压力大, 生活节奏快, 焦虑, 工作压力, 职业倦怠, 状态, 对此, 打工人, 职场...   

                                 Representative_Docs  
0  [，山东大学儒学高等研究院黄玉顺教授莅临华东师范大学哲学系进行学术交流。其间，对黄玉顺教授进...  
1  [为促进纪检监察干部身心健康、缓解工作压力，讲座特别邀请南县中医医院副主任医师刘宇专题授课，...  
2  [，出版品牌影响力大会在广西桂林靖江王府国学堂举行。作为中国出版界品牌建设的年度标杆盛会，本...  
3  ["保肝"兴起的背后生活方式的改变如今生活节奏快，很多年轻人长期熬夜、饮酒，工作压力大。肝脏...  


In [27]:
# 10 写回 Excel
# =========================

df_filtered["topic"] = topics_reduced

# 自动写入 topic keywords

topic_labels = {
    topic: ", ".join([word for word, _ in topic_model.get_topic(topic)])
    for topic in topic_model.get_topics().keys()
}

df_filtered["topic_keywords"] = df_filtered["topic"].map(topic_labels)

import os

output_path = os.path.join(os.path.expanduser("~/Desktop"), "news_topic_results.xlsx")

df_filtered.to_excel(output_path, index=False)

print("结果已保存:", output_path)

结果已保存: /Users/lixiangzhen/Desktop/news_topic_results.xlsx


In [28]:
# 11 可视化
# =========================


import os

out_path = os.path.expanduser("~/Desktop/news_topics_visualization.html")
try:
    fig = topic_model.visualize_topics()
    fig.write_html(out_path)
    print("已保存主题 2D 图:", out_path)
except ValueError as e:
    if "zero-size array" in str(e):
        print("提示：当前主题数过少，visualize_topics() 无法做 UMAP，改用条形图。")
        fig = topic_model.visualize_barchart()
        fig.write_html(out_path)
        print("已保存主题条形图:", out_path)
    else:
        raise
print("BERTopic 完成")

已保存主题 2D 图: /Users/lixiangzhen/Desktop/news_topics_visualization.html
BERTopic 完成
